In [1]:
import json
import uuid
from typing import Annotated, Literal, Sequence, TypedDict

from langchain_core.messages import (
    AIMessage,
    BaseMessage,
    HumanMessage,
    ToolCall,
    ToolMessage,
)
from langchain_core.prompts.chat import ChatPromptTemplate
from langchain_core.runnables import RunnableConfig, RunnableLambda
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages
from toolbox_langchain import ToolboxTool
from typing import Any, Dict, List, Optional, Sequence
from toolbox_langchain import ToolboxClient

In [2]:
def __is_logged_in(config: RunnableConfig) -> bool:
    return bool(
        config
        and "configurable" in config
        and "auth_token_getters" in config["configurable"]
        and "my_google_service" in config["configurable"]["auth_token_getters"]
        and config["configurable"]["auth_token_getters"]["my_google_service"]()
    )

In [3]:
def __get_tool_to_run(tool: ToolboxTool, config: RunnableConfig):
    if (
        config
        and "configurable" in config
        and "auth_token_getters" in config["configurable"]
    ):
        auth_token_getters = config["configurable"]["auth_token_getters"]
        if auth_token_getters:
            core_tool = tool._ToolboxTool__core_tool  # type: ignore
            required_auth_keys = set(core_tool._required_authz_tokens)
            for auth_list in core_tool._required_authn_params.values():
                required_auth_keys.update(auth_list)
            filtered_getters = {
                k: v for k, v in auth_token_getters.items() if k in required_auth_keys
            }
            if filtered_getters:
                return tool.add_auth_token_getters(filtered_getters)
    return tool

In [4]:
TOOLBOX_URL = "http://127.0.0.1:5000"
client = ToolboxClient(
    TOOLBOX_URL, 
    # client_headers={"Authorization": auth_token_provider}
)
tools = await client.aload_toolset("cymbal_air")

In [5]:
tools[1]

ToolboxTool(name='list_flights', description='Use this tool to list flights information matching search criteria.\nTakes an arrival airport, a departure airport, or both, filters by date and returns all matching flights.\nIf 3-letter iata code is not provided for departure_airport or arrival_airport, use `search_airports` tool to get iata code information.\nDo NOT guess a date, ask user for date input if it is not given. Date must be in the following format: YYYY-MM-DD.\nThe agent can decide to return the results directly to the user.\n\nArgs:\n    arrival_airport (Optional): Arrival airport 3-letter code\n    date (str): Date of flight departure\n    departure_airport (Optional): Departure airport 3-letter code', args_schema=<class 'toolbox_core.utils.list_flights'>)

### Testing

In [6]:
class UserState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

async def node_1(state: UserState, config: RunnableConfig):
    print("⚙️ [NODE 1] Running...")
    print(f"Logged in: {__is_logged_in(config)}")
    return {"messages": []}    

async def node_2(state: UserState, config: RunnableConfig):
    print("⚙️ [NODE 2] Running...")
    __list_flights = __get_tool_to_run(tools[1], config)

    tool_args = {
        "departure_airport": "SFO",
        "arrival_airport": "DEN",
        "date": "01-01-2025",
    }
    output = await __list_flights.ainvoke(tool_args)
    print(output)
    return {"messages": []}

NODE_1 = "node_1"
NODE_2 = "node_2"

llm_graph = StateGraph(UserState)

# NODES
llm_graph.add_node(NODE_1, RunnableLambda(node_1))
llm_graph.add_node(NODE_2, RunnableLambda(node_2))

# EDGES
llm_graph.add_edge(NODE_1, NODE_2)
llm_graph.add_edge(NODE_2, END)

# START
llm_graph.set_entry_point(NODE_1)

# COMPILE
checkpointer = MemorySaver()
langgraph_app = llm_graph.compile(
    checkpointer=checkpointer, 
    debug=False, 
)

In [10]:
def get_user_id_token() -> Optional[str]:
    # return None
    return "simple-uid"

In [11]:
config = {
    "configurable": {
        "thread_id": "test-id",
        "checkpoint_ns": "",
        "auth_token_getters": {
            "my_google_service": lambda: get_user_id_token()
        },
    },
}

In [12]:
response = await langgraph_app.ainvoke(
    {
        "messages": HumanMessage(content="Test"),
    },
    config=config,
) 

⚙️ [NODE 1] Running...
Logged in: True
⚙️ [NODE 2] Running...
[{"id":0,"airline":"UA","flight_number":"1532","departure_airport":"SFO","arrival_airport":"DEN","departure_time":"2025-01-01T05:50:00Z","arrival_time":"2025-01-01T09:23:00Z","departure_gate":"E49","arrival_gate":"D6"},{"id":59,"airline":"UA","flight_number":"720","departure_airport":"SFO","arrival_airport":"DEN","departure_time":"2025-01-01T12:46:00Z","arrival_time":"2025-01-01T16:19:00Z","departure_gate":"C41","arrival_gate":"E39"},{"id":84,"airline":"UA","flight_number":"1733","departure_airport":"SFO","arrival_airport":"DEN","departure_time":"2025-01-01T14:38:00Z","arrival_time":"2025-01-01T18:35:00Z","departure_gate":"B27","arrival_gate":"E32"},{"id":139,"airline":"F9","flight_number":"664","departure_airport":"SFO","arrival_airport":"DEN","departure_time":"2025-01-01T19:31:00Z","arrival_time":"2025-01-01T23:13:00Z","departure_gate":"E11","arrival_gate":"E12"},{"id":142,"airline":"UA","flight_number":"1243","departure_a